# IDSIA Melon results

Baselines are loaded, not trained: analytic physics and the authors' released physics+residual checkpoint (`phys+res_random_square_chirp.pt`). Our trained families load from `runs/quadrotor/` (or `idsia/out/runs/`). A cached `out/eval/` MAE is reused only if it is newer than its checkpoint. MAE at h=1 / 10 / 50.

In [ ]:
import json

from quadrotor.geometry import train_device
from quadrotor.evaluate import benchmark_mae
from quadrotor.idsia.data import load_idsia_prepared
from quadrotor.idsia.eval.baselines import load_baseline
from quadrotor.idsia.eval.evaluate import eval_model, family_summary, pack_mae, save_eval
from quadrotor.idsia.train.lib import list_seeds, load_trained
from quadrotor.idsia.utils.idsia_training_reproduction_params import BASELINES, EVAL, HORIZON, KEYS, TRAIN_FAMILIES, run_dir

FAMILY = "all"  # or "phys_quotient" / "phys_quotient_sym" / "sym_quad_dynamics"
DEVICE = train_device("mps")
_train, _val, test = load_idsia_prepared()
families = TRAIN_FAMILIES if FAMILY == "all" else (FAMILY,)

In [ ]:
def cell(values):
    return " / ".join(f"{v:.4f}" for v in values)


def show(name, packed):
    print(f"{name:22s}  " + "  ".join(f"{cell(packed[k]):28s}" for k in KEYS))


print(f"{'model':22s}  {'p':28s}  {'R':28s}  {'v':28s}  {'ω'}")
for name in BASELINES:  # physics, published phys+res
    mae = benchmark_mae(load_baseline(name, DEVICE), test, horizon=HORIZON, start_stride=1)
    save_eval(name, 0, mae)
    show(name, pack_mae(mae))

for family in families:
    seeds = list_seeds(family)
    if not seeds:
        print(f"{family:22s}  no checkpoints")
        continue
    for seed in seeds:
        mae_path = EVAL / family / f"seed_{seed}" / "mae.json"
        ckpt_path = run_dir(family, seed) / "best.pt"
        # Reuse the cached MAE only if it is newer than the checkpoint it scores.
        if mae_path.is_file() and mae_path.stat().st_mtime >= ckpt_path.stat().st_mtime:
            row = json.loads(mae_path.read_text())
            packed = {k: row[k] for k in KEYS}
        else:
            model = load_trained(family, seed, device=DEVICE)
            packed = eval_model(model, test, family, seed)
        show(f"{family} s{seed}", packed)
    if len(seeds) > 1:
        summary = family_summary(family)
        show(f"{family} mean", {k: summary[k]["mean"] for k in KEYS})